## Ch11. Hierarchical and grouped time series Forecasting: Principles & Practice (Python Edition) Extracted from: fpppy-11-hierarchical-forecasting.qmd

## [Slide 3] Australian Tourism Example

In [ ]:
import pandas as pd
from hierarchicalforecast.utils import aggregate

spec = [
    ["Country"],
    ["Country", "State"],
    ["Country", "State", "Region"],
]
Y_df, S_df, tags = aggregate(
    df=aus_tourism.drop(columns=["unique_id"]),
    spec=spec
)

## [Slide 6] Australian Prison Population

In [ ]:
prison = pd.read_csv("data/prison.csv", index_col=0).assign(
    ds=lambda x: pd.PeriodIndex(x["ds"], freq="Q").to_timestamp(),
    Country="Australia",
)

spec = [
    ["Country"],
    ["Country", "State"],
    ["Country", "Gender"],
    ["Country", "Legal"],
    ["Country", "State", "Gender"],
    ["Country", "State", "Legal"],
    ["Country", "Legal", "Gender"],
    ["Country", "State", "Legal", "Gender"],
]
Y_df, S_df, tags = aggregate(df=prison, spec=spec)

## [Slide 8] Mixed Structure: Tourism with Purpose

In [ ]:
spec = [
    ["Country", "State"],
    ["Country", "Purpose"],
    ["Country", "State", "Purpose"],
    ["Country", "State", "Region", "Purpose"],
]
Y_df, S_df, tags = aggregate(
    df=aus_tourism.drop(columns=["unique_id"]),
    spec=spec,
)

## [Slide 11] Bottom-Up: Code

In [ ]:
from hierarchicalforecast.core import HierarchicalReconciliation
from hierarchicalforecast.methods import BottomUp
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

reconcilers = [BottomUp()]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)

sf = StatsForecast(
    models=[AutoETS(season_length=4)],
    freq="Q", n_jobs=-1
)
Y_hat_df = sf.forecast(h=4, df=Y_df)

Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df,
    Y_df=Y_df,
    S_df=S_df,
    tags=tags
)

## [Slide 14] Top-Down: Code

In [ ]:
from hierarchicalforecast.methods import TopDown

Method 1: average historical proportions

In [ ]:
reconcilers = [TopDown(method="average_proportions")]

Method 2: proportions of historical averages

In [ ]:
reconcilers = [TopDown(method="proportion_averages")]

Method 3: forecast proportions (recommended)

In [ ]:
reconcilers = [TopDown(method="forecast_proportions")]

hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df,
    Y_df=Y_df,
    S_df=S_df,
    tags=tags
)

## [Slide 16] Middle-Out Approach

In [ ]:
from hierarchicalforecast.methods import MiddleOut

reconcilers = [
    MiddleOut(
        middle_level=2,
        top_down_method="forecast_proportions"
    )
]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df, Y_df=Y_df,
    S_df=S_df, tags=tags
)

## [Slide 23] MinT Code

In [ ]:
from hierarchicalforecast.methods import MinTrace

reconcilers = [
    BottomUp(),
    MinTrace(method="ols"),
    MinTrace(method="wls_var"),
    MinTrace(method="wls_struct"),
    MinTrace(method="mint_shrink"),  # recommended
]

hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df,
    Y_df=Y_fitted_df,   # needs fitted values for W estimation
    S_df=S_df,
    tags=tags
)

## [Slide 25] Full Workflow

In [ ]:
from statsforecast.models import AutoETS

spec = [
    ["Country"],
    ["Country", "State"],
    ["Country", "Purpose"],
    ["Country", "State", "Region"],
    ["Country", "State", "Purpose"],
    ["Country", "State", "Region", "Purpose"],
]
Y_df, S_df, tags = aggregate(
    aus_tourism.drop(columns=["unique_id"]), spec)

Y_train_df = Y_df.loc[lambda x: x["ds"] < "2016"]
Y_test_df  = Y_df.loc[lambda x: x["ds"] >= "2016"]

sf = StatsForecast(
    models=[AutoETS(season_length=4)],
    freq="Q", n_jobs=-1
)
Y_hat_df    = sf.forecast(h=8, df=Y_train_df, fitted=True)
Y_fitted_df = sf.forecast_fitted_values()

## [Slide 27] Reconciliation and Evaluation

In [ ]:
from hierarchicalforecast.evaluation import evaluate
from utilsforecast.losses import rmse, mase
from functools import partial

reconcilers = [
    BottomUp(),
    MinTrace(method="ols"),
    MinTrace(method="mint_shrink"),
]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df, Y_df=Y_fitted_df,
    S_df=S_df, tags=tags
)

eval_tags = {
    "Total":   tags["Country"],
    "Purpose": tags["Country/Purpose"],
    "State":   tags["Country/State"],
    "Regions": tags["Country/State/Region"],
    "Bottom":  tags["Country/State/Region/Purpose"],
}
eval_df = Y_rec_df.merge(Y_test_df, on=["unique_id", "ds"])
evaluation = evaluate(
    df=eval_df, tags=eval_tags, train_df=Y_train_df,
    metrics=[rmse, partial(mase, seasonality=4)],
)

## [Slide 29] Coherent Probabilistic Forecasts

In [ ]:
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df, Y_df=Y_fitted_df,
    S_df=S_df, tags=tags,
    intervals_method='normality'
)

## [Slide 31] Bootstrap Approach

In [ ]:
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df, Y_df=Y_fitted_df,
    S_df=S_df, tags=tags,
    intervals_method='bootstrap'
)

from utilsforecast.losses import scaled_crps, mqloss

evaluation = evaluate(
    df=eval_df, tags=eval_tags, train_df=Y_train_df,
    metrics=[partial(mase, seasonality=4), mqloss],
    level=list(range(10, 100, 10)),
)

## [Slide 33] Grouped Structure Application

In [ ]:
from statsforecast.models import Naive

spec = [
    ["Country"],
    ["Country", "State"],
    ["Country", "Gender"],
    ["Country", "Legal"],
    ["Country", "State", "Gender", "Legal"],
]
prison_scaled = prison.assign(y=prison["y"] / 1e3)
Y_df, S_df, tags = aggregate(prison_scaled, spec)

Y_train_df = Y_df.loc[lambda x: x["ds"] < "2015"]
Y_test_df  = Y_df.loc[lambda x: x["ds"] >= "2015"]

models = [AutoETS(season_length=4, model="MAM"), Naive()]
sf = StatsForecast(models=models, freq="QS", n_jobs=-1)

## [Slide 35] Prison Population: Reconciliation

In [ ]:
levels = list(range(10, 100, 10))
sf.fit(df=Y_train_df)
Y_hat_df    = sf.forecast(h=8, df=Y_train_df,
                           fitted=True, level=levels)
Y_fitted_df = sf.forecast_fitted_values()

reconcilers = [
    BottomUp(),
    MinTrace(method="mint_shrink"),
]
hrec = HierarchicalReconciliation(reconcilers=reconcilers)
Y_rec_df = hrec.reconcile(
    Y_hat_df=Y_hat_df, Y_df=Y_fitted_df,
    S_df=S_df, tags=tags, level=levels
)